# Overview

This notebook corresponds to makemore part 3.


In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Load Data and Build Datasets

Removed checks and EDA (covered in part 2 notebook)


In [ ]:
with open("names.txt", "r") as file:
    names = file.read().splitlines()

In [ ]:
chars = ["."] + sorted(set("".join(names)))
s_to_i = {s: i for i, s in enumerate(chars)}
i_to_s = {i: s for i, s in enumerate(chars)}

vocab_size = len(chars)


def build_dataset(names, block_size=3):
    X, y = [], []
    for name in names:
        context = [0] * block_size
        name += "."
        for ch in name:
            idx = s_to_i[ch]
            X.append(context)
            y.append(idx)
            context = context[1:] + [idx]

    return torch.tensor(X), torch.tensor(y)

In [ ]:
random.seed(42)
random.shuffle(names)

p1, p2 = int(len(names) * 0.8), int(len(names) * 0.9)

block_size = 3

X_train, y_train = build_dataset(names[:p1], block_size)
X_val, y_val = build_dataset(names[p1:p2], block_size)
X_test, y_test = build_dataset(names[p2:], block_size)

# Build MLP


PyTorch-esque classes


In [ ]:
g = torch.Generator().manual_seed(2147483647)


class Linear:
    def __init__(self, in_features, out_features, bias=True):
        self.weights = (
            torch.randn((in_features, out_features), generator=g) / in_features**0.5
        )
        self.bias = torch.zeros(out_features) if bias else None

    def __call__(self, input):
        result = input @ self.weights
        if self.bias is not None:
            result += self.bias

        return result

    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, num_features, eps=1e-5, momentum=1e-3):
        self.eps = eps
        # Changed default momentum to be better for this context
        self.momentum = momentum

        self.gamma = torch.ones(num_features)
        self.beta = torch.zeros(num_features)

        self.running_mean = torch.zeros(num_features)
        self.running_var = torch.ones(num_features)

        self.training = True

    def __call__(self, input):
        if self.training:
            input_mean = input.mean(dim=0)
            input_var = input.var(dim=0)

            with torch.no_grad():
                self.running_mean = (
                    1 - self.momentum
                ) * self.running_mean + self.momentum * input_mean
                self.running_var = (
                    1 - self.momentum
                ) * self.running_var + self.momentum * input_var

            return (
                self.gamma * ((input - input_mean) / torch.sqrt(input_var + self.eps))
                + self.beta
            )
        else:
            return (
                self.gamma
                * (
                    (input - self.running_mean)
                    / torch.sqrt(self.running_var + self.eps)
                )
                + self.beta
            )

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __init__(self):
        return

    def __call__(self, input):
        return torch.tanh(input)

    def parameters(self):
        return []

Initialize model


In [ ]:
emb_dim = 10
hidden_size = 100

# fmt: off
C = torch.randn((vocab_size, emb_dim), generator=g)
layers = [
    Linear(emb_dim * block_size, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    Linear(hidden_size, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    Linear(hidden_size, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    Linear(hidden_size, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    Linear(hidden_size, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    Linear(hidden_size, vocab_size, bias=False), BatchNorm1d(vocab_size),
]
# fmt: on

with torch.no_grad():
    # Make predictions less confident at the start
    layers[-1].gamma *= 0.1
    # Proper Kaiming initialization
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weights *= 5 / 3

params = [C] + [p for layer in layers for p in layer.parameters()]
for p in params:
    p.requires_grad = True

print(f"Total params: {sum(p.numel() for p in params)}")

# Train and Evaluate MLP


In [ ]:
batch_size = 256
num_steps = 200000
lrs = [0.02, 0.2, 0.02, 0.002]
milestones = [2000, 120000, 150000]

# Reset generator for reproducibility
g = torch.Generator().manual_seed(42)

losses = []
j = 0
for i in range(num_steps):
    # Get mini-batch
    idxs = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    X_batch, y_batch = X_train[idxs], y_train[idxs]

    # Forward pass
    embs = C[X_batch]
    x = embs.reshape((len(X_batch), -1))
    for layer in layers:
        if isinstance(layer, BatchNorm1d):
            layer.training = True
        x = layer(x)

    # Compute loss
    loss = F.cross_entropy(x, y_batch)

    # Backward pass

    # Get gradients
    for p in params:
        p.grad = None
    loss.backward()

    # Set LR
    if j < len(milestones) and i >= milestones[j]:
        j += 1
    lr = lrs[j]

    # GD step
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    losses.append(loss.item())

    if i % 10000 == 0:
        print(f"Epoch: {i}/{num_steps}, loss: {loss}")


In [ ]:
plt.plot(losses)

In [ ]:
def evaluate(X, y):
    embs = C[X]
    x = embs.reshape((len(X), -1))
    for layer in layers:
        if isinstance(layer, BatchNorm1d):
            layer.training = False
        x = layer(x)

    return F.cross_entropy(x, y).item()

In [ ]:
train_loss = evaluate(X_train, y_train)
val_loss = evaluate(X_val, y_val)

print(f"Train loss: {train_loss}")
print(f"Val loss: {val_loss}")

Val loss:

- Without BN: 2.09651327
- With BN: 2.08763897


# Inference


In [ ]:
for _ in range(5):
    context = [0] * block_size
    out = ""
    while True:
        embs = C[torch.tensor(context)]
        x = embs.reshape(1, -1)
        for layer in layers:
            if isinstance(layer, BatchNorm1d):
                layer.training = False
            x = layer(x)

        probs = F.softmax(x, dim=1)
        idx = torch.multinomial(probs, num_samples=1).item()
        if idx == 0:
            break
        context = context[1:] + [idx]
        out += i_to_s[idx]

    print(out)
